# Multi-Turn Conversation Eval

[Read the step-by-step guide](https://docs.futureagi.com/docs/cookbook/quickstart/conversation-eval). Compare how two support agents handle the same duplicate charge. Run the cells in order in a fresh Python 3.12 kernel.

## Install

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "agent-learning-kit==0.3.7"], check=True)

In [ ]:
import os
from getpass import getpass

for name in ("FI_API_KEY", "FI_SECRET_KEY"):
    if not os.environ.get(name):
        value = getpass(f"{name}: ")
        if not value:
            raise ValueError(f"{name} is required")
        os.environ[name] = value
        del value

## Set up the evaluator

Cache results so each metric runs only once per transcript.

In [ ]:
import json
from pathlib import Path
from fi.evals import Evaluator

evaluator = Evaluator()
results = {}


def evaluate(metric, label, conversation, **extra_inputs):
    key = (metric, label)
    if key not in results:
        result = evaluator.evaluate(
            eval_templates=metric,
            inputs={"conversation": json.dumps(conversation), **extra_inputs},
            model_name="turing_small",
        ).eval_results[0]
        if not result.eval_id or result.output is None:
            raise RuntimeError(f"{metric} returned no score: {result.reason}")
        results[key] = result
    return results[key]


def verdict(result):
    output = result.output
    return output["choice"] if isinstance(output, dict) else output

## Define two conversations

In [ ]:
good_conversation = [
    {"role": "user", "content": "Hi, I was charged twice for order #7291. Can you help?"},
    {"role": "assistant", "content": "I'm sorry about that! Let me look up order #7291 right away."},
    {"role": "assistant", "content": "I can see the duplicate charge. I've initiated a refund of $49.99 to your original payment method. It should appear within 3-5 business days."},
    {"role": "user", "content": "Great, thank you. Will I get a confirmation email?"},
    {"role": "assistant", "content": "Yes, you'll receive a refund confirmation email within the next hour. Is there anything else I can help with?"},
    {"role": "user", "content": "No, that's all. Thanks for the quick help!"},
    {"role": "assistant", "content": "You're welcome! Don't hesitate to reach out if you need anything else. Have a great day!"},
]

bad_conversation = [
    {"role": "user", "content": "Hi, I was charged twice for order #7291. Can you help?"},
    {"role": "assistant", "content": "I'd be happy to help! Have you tried checking your account settings?"},
    {"role": "user", "content": "This isn't about account settings. I was charged $49.99 twice for the same order."},
    {"role": "assistant", "content": "I understand. Could you provide your order number?"},
    {"role": "user", "content": "I already said it's order #7291."},
    {"role": "assistant", "content": "Could you provide your order number so I can look into this?"},
    {"role": "user", "content": "I just told you twice, #7291. Can I speak to a manager?"},
    {"role": "assistant", "content": "I apologize for the inconvenience. Have you considered upgrading to our Premium support plan for faster resolution?"},
    {"role": "user", "content": "No! I just want my refund. Please transfer me to someone who can help."},
    {"role": "assistant", "content": "I understand your concern. Let me check your account settings to see if there's an issue."},
]

## Score overall conversation quality

In [ ]:
for label, convo in [("Good", good_conversation), ("Bad", bad_conversation)]:
    result = evaluate("customer_agent_conversation_quality", label, convo)
    print(f"{label} conversation: {verdict(result)}/5")
    print(f"  Reason: {result.reason}\n")

## Diagnose why the bad conversation failed

### Context retention

In [ ]:
result = evaluate("customer_agent_context_retention", "Bad", bad_conversation)
print(f"Context retention: {verdict(result)}")
print(f"Reason: {result.reason}")

### Query handling

In [ ]:
result = evaluate("customer_agent_query_handling", "Bad", bad_conversation)
print(f"Query handling: {verdict(result)}")
print(f"Reason: {result.reason}")

### Loop detection

In [ ]:
result = evaluate("customer_agent_loop_detection", "Bad", bad_conversation)
print(f"Loop detection: {verdict(result)}")
print(f"Reason: {result.reason}")

### Human escalation

In [ ]:
result = evaluate("customer_agent_human_escalation", "Bad", bad_conversation)
print(f"Human escalation: {verdict(result)}")
print(f"Reason: {result.reason}")

## Evaluate prompt conformance

In [ ]:
system_prompt = (
    "You are a billing support agent for Ridgeline Outfitters. "
    "Your role is to help customers resolve payment and billing issues. "
    "Always acknowledge the customer's issue first, then investigate. "
    "Never upsell products during a support interaction. "
    "If a customer asks to speak with a manager, escalate immediately."
)

for label, convo in [("Good", good_conversation), ("Bad", bad_conversation)]:
    result = evaluate("customer_agent_prompt_conformance", label, convo, system_prompt=system_prompt)
    print(f"{label} conversation - prompt conformance: {verdict(result)}")
    print(f"  Reason: {result.reason}\n")

## Run a full scorecard

Compare the two transcripts side by side. `conversation-scorecard.json` saves each verdict with its execution ID.

In [ ]:
metrics = [
    ("conversation_coherence", "Coherence"),
    ("conversation_resolution", "Resolution"),
    ("customer_agent_conversation_quality", "Quality"),
    ("customer_agent_context_retention", "Context"),
    ("customer_agent_query_handling", "Queries"),
    ("customer_agent_loop_detection", "Loops"),
    ("customer_agent_human_escalation", "Escalation"),
]

print(f"{'Metric':<14}  {'Good':>12}  {'Bad':>12}")
print("-" * 42)

for metric_name, label in metrics:
    good_val = verdict(evaluate(metric_name, "Good", good_conversation))
    bad_val = verdict(evaluate(metric_name, "Bad", bad_conversation))
    print(f"{label:<14}  {str(good_val):>12}  {str(bad_val):>12}")

# Keep the execution IDs with their verdicts for later lookup.
scorecard = [
    {"metric": metric, "case": case, "id": result.eval_id, "verdict": verdict(result)}
    for (metric, case), result in results.items()
]
Path("conversation-scorecard.json").write_text(json.dumps(scorecard, indent=2))

## Evaluate more conversations in the dashboard

For a dataset, map one transcript per row to `conversation`. Prompt conformance also needs `system_prompt`.